copying and adjusting code from here: https://huggingface.co/spaces/hesamation/primer-llm-embedding?section=embeddings_in_action_%28deepseek-r1-distill-qwen-1.5b%29

In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
# Define some example sentences
sentences = [
    "The cat sat on the mat.",
    "The dog slept on the floor.",
    "I love natural language processing."
]

# Load a pre-trained model and generate embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
dataset_embeddings = model.encode(sentences)

# Get dataset_embeddings with mean pooling
print(f"Embedding shape: {dataset_embeddings.shape}")
print(f"First 5 dimensions of the sentences' embeddings:\n{np.round(dataset_embeddings[:, :5], 3)}")

# Calculate cosine similarity between the first two sentences
similarity = cosine_similarity([dataset_embeddings[0]], [dataset_embeddings[1]])
print(f"Cosine similarity between '{sentences[0]}' and '{sentences[1]}': {np.round(similarity[0][0], 3)}")

Embedding shape: (3, 384)
First 5 dimensions of the sentences' embeddings:
[[ 0.13  -0.016 -0.037  0.058 -0.06 ]
 [ 0.01  -0.01  -0.039  0.14  -0.006]
 [ 0.039 -0.078  0.055  0.     0.036]]
Cosine similarity between 'The cat sat on the mat.' and 'The dog slept on the floor.': 0.40799999237060547


In [3]:
dataset_embeddings.shape

(3, 384)

In [4]:
from sentence_transformers.util import semantic_search

query = "A cat is on a mat."
query_embeddings = model.encode([query])

hits = semantic_search(query_embeddings, dataset_embeddings, top_k=2)
hits

[[{'corpus_id': 0, 'score': 0.9035928249359131},
  {'corpus_id': 1, 'score': 0.3671172559261322}]]

In [5]:
import sqlite3
import sqlite_vec

In [6]:
# Connect to VVZ SQLite database
conn = sqlite3.connect('lectures_FS2026.db')
c_lect = conn.cursor()

# Connect to embedding database (or create it if it doesn't exist)
conn_emb = sqlite3.connect('embeddings_FS2026.db')
c_emb = conn_emb.cursor()

# load the sqlite vector extension
conn_emb.enable_load_extension(True)
sqlite_vec.load(conn_emb)
conn_emb.enable_load_extension(False)

vec_version, = conn_emb.execute("select vec_version()").fetchone()
print(f"vec_version={vec_version}")

vec_version=v0.1.6


In [7]:
# create a table to store embeddings if it doesn't exist

c_emb.execute('DROP TABLE IF EXISTS vss_embeddings')
c_emb.execute('DROP TABLE IF EXISTS embeddings')
c_emb.execute('''
CREATE TABLE IF NOT EXISTS embeddings (
    id INTEGER PRIMARY KEY,
    lecture_number TEXT,
    embedding BLOB
)
''')

c_emb.execute('''
CREATE VIRTUAL TABLE IF NOT EXISTS vss_embeddings USING vec0 (
    id INTEGER PRIMARY KEY,
    lecture_number TEXT,
    embedding float[384]
)
''')

In [8]:
# get lecture data
c_lect.execute('SELECT number, title, abstract, learning_objective, content, lecture_notes, literature, performance_assessment FROM lectures')
lectures = c_lect.fetchall()
len(lectures)

2544

In [9]:
# remove duplicates
lectures = list(set(lectures))
print(f"Total unique lectures: {len(lectures)}")

Total unique lectures: 2544


In [10]:
lectures[0:4]

[('401-5650-00L',
  'Zurich Colloquium in Applied and Computational Mathematics',
  'Research colloquium',
  None,
  None,
  None,
  None),
 ('102-0214-01L',
  'Excursion - Water Supply of Vienna',
  'How does the Viennese water supply work? View the facilities and the fundamentals of the physical region: from the urban area storages to the catchment area springs in the Styria, taking account of scientific research projects in the context of karst research (geology, hydrology, karst hydrology, biology/vegetation, forest ecology, snow measurement programs)',
  'The participants of the excursion learn:1. How the Viennese water supply works.2. Which constructions and facilities are required to operate such a large water supply, especially in the case of the Viennese spring water supply (1st and 2nd Viennese spring water main)- reservoir "Rosenhügel" (in the city area)- the biggest closed reservoir of Europe "Neusiedl am Steinfeld" along the 1st Viennese spring water main (out of city)- 1s

In [11]:
def has_meaningful_content(text):
    if not text:
        return False
    text = text.strip()
    if len(text) < 20:  # skip trivial fields
        return False
    return True


batch_size = 16
for i in range(0, len(lectures), batch_size):
    batch = lectures[i:i+batch_size]

    texts = []
    lecture_numbers = []

    for lecture in batch:
        number, title, abstract, learning_objective, content, lecture_notes, literature, performance_assessment = lecture

        for text in [title, abstract, learning_objective, content, lecture_notes, literature, performance_assessment]:
            if has_meaningful_content(text):
                texts.append(text)
                lecture_numbers.append(number)

    if not texts:
        continue

    embeddings = model.encode(texts)

    # insert into database
    for lecture_number, embedding in zip(lecture_numbers, embeddings):
        # Insert into embeddings table
        c_emb.execute('INSERT INTO embeddings (lecture_number, embedding) VALUES (?, ?)', 
                        (lecture_number, embedding.tobytes()))
        
        # Insert into vector search table
        c_emb.execute('INSERT INTO vss_embeddings (lecture_number, embedding) VALUES (?, ?)', 
                        (lecture_number, embedding))

    # Commit after each batch
    conn_emb.commit()
    print(f"Processed batch {i//batch_size + 1}")


Processed batch 1
Processed batch 2
Processed batch 3
Processed batch 4
Processed batch 5
Processed batch 6
Processed batch 7
Processed batch 8
Processed batch 9
Processed batch 10
Processed batch 11
Processed batch 12
Processed batch 13
Processed batch 14
Processed batch 15
Processed batch 16
Processed batch 17
Processed batch 18
Processed batch 19
Processed batch 20
Processed batch 21
Processed batch 22
Processed batch 23
Processed batch 24
Processed batch 25
Processed batch 26
Processed batch 27
Processed batch 28
Processed batch 29
Processed batch 30
Processed batch 31
Processed batch 32
Processed batch 33
Processed batch 34
Processed batch 35
Processed batch 36
Processed batch 37
Processed batch 38
Processed batch 39
Processed batch 40
Processed batch 41
Processed batch 42
Processed batch 43
Processed batch 44
Processed batch 45
Processed batch 46
Processed batch 47
Processed batch 48
Processed batch 49
Processed batch 50
Processed batch 51
Processed batch 52
Processed batch 53
Pr

In [12]:
embeddings.shape

(66, 384)

### Testing insertion and retrieval from embedding database

In [ ]:
# embedding = np.random.rand(384).astype('float32')
# lecture_number = "Lecture_1"

# c_emb.execute('INSERT INTO vss_embeddings (lecture_number, embedding) VALUES (?, ?)', 
#                         (lecture_number, embedding))
# conn_emb.commit()

In [13]:
# Define the target embedding (e.g., from a query or another source)
# target_embedding = np.random.rand(384).astype('float32')

lecture_name = "Introduction to Machine Learning"
target_embedding = model.encode([lecture_name])

# Perform a similarity search in the virtual table
query = '''
SELECT lecture_number, distance
FROM vss_embeddings
WHERE embedding MATCH ?
AND k = 5
'''

# Execute the query with the target embedding
results = c_emb.execute(query, (target_embedding,)).fetchall()

# Print the results
for lecture_number, similarity in results:
    print(f"Lecture Number: {lecture_number}, Similarity: {similarity}")

# get the lecture from the lecture number
for lecture_number, _ in results:
    c_lect.execute('SELECT number, title, abstract, learning_objective, content, lecture_notes, literature, performance_assessment FROM lectures WHERE number=?', (lecture_number,))
    lecture = c_lect.fetchone()
    print(lecture)
    print("\n---\n")

Lecture Number: 252-0220-00L, Similarity: 1.0140523727386608e-06
Lecture Number: 441-1000-00L, Similarity: 0.6481214165687561
Lecture Number: 103-0849-AAL, Similarity: 0.6555092334747314
Lecture Number: 103-0849-00L, Similarity: 0.6555092334747314
Lecture Number: 441-1000-00L, Similarity: 0.7519223690032959
('252-0220-00L', 'Introduction to Machine Learning', 'The course introduces the foundations of learning and making predictions based on data.', 'The course will introduce the foundations of learning and making predictions from data. We will study basic concepts such as trading goodness of fit and model complexitiy. We will discuss important machine learning algorithms used in practice, and provide hands-on experience in a course project.', "- Linear regression (overfitting, cross-validation/bootstrap, model selection, regularization, [stochastic] gradient descent)- Linear classification: Logistic regression (feature selection, sparsity, multi-class)- Kernels and the kernel trick (Pr

In [14]:
# close the connections
conn.close()
conn_emb.close()